### 15-06-2026

In [4]:
# 01_eda_initial.py

%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import gc

# Hardcoded absolute path to bypass any notebook folder path traps
PROJECT_ROOT = '/Users/abannee/Documents/GitHub/fraud_detection_ml/'

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data_utils import load_and_merge_data
from src.data_utils import engineer_time_features

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw')
TRAIN_TXN = os.path.join(DATA_DIR, 'train_transaction.csv')
TRAIN_ID = os.path.join(DATA_DIR, 'train_identity.csv')

df = load_and_merge_data(TRAIN_TXN, TRAIN_ID)
df = engineer_time_features(df)

Loading transaction dataset...
Loading identity dataset...
Executing defensive Left-Merge across TransactionID fields...
Memory usage of dataframe is 1984.20 MB
Memory usage after optimization is: 1073.53 MB
Decreased by 45.9%


/Users/abannee/Documents/GitHub/fraud_detection_ml/src/data_utils.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Hour'] = (df['TransactionDT'] // 3600) % 24
/Users/abannee/Documents/GitHub/fraud_detection_ml/src/data_utils.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['DayOfWeek'] = (df['TransactionDT'] // (3600 * 24)) % 7


In [6]:
# Constructing card proxy identifier combinations to pinpoint distinct payment components securely
print("Generating high-cardinality entity proxy tracking keys...")
df['card_proxy_id'] = df['card1'].astype(str) + "_" + df['card2'].astype(str) + "_" + df['addr1'].astype(str)

print("Enforcing strict chronological index split arrays...")
df = df.sort_values('TransactionDT').reset_index(drop=True)

# 70-30 Sequential Split Boundary Lookups
split_idx = int(len(df) * 0.70)
train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()

del df
gc.collect()

Generating high-cardinality entity proxy tracking keys...


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_75208/3216402972.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['card_proxy_id'] = df['card1'].astype(str) + "_" + df['card2'].astype(str) + "_" + df['addr1'].astype(str)


Enforcing strict chronological index split arrays...


0

In [9]:
# =====================================================================
# SYSTEM DEFENSE LOGIC: CALCULATION ENTIRELY ANCHORED TO TRAIN WINDOW
# =====================================================================
print("\n--- Engineering Advanced Ratio Features ---")
# Group metrics computed STRICTLY on train records to fully block operational look-ahead leakages
card_mean_amt_map = train_df.groupby('card_proxy_id')['TransactionAmt'].mean().to_dict()

# Map the statistics safely to the cohorts
train_df['card_mean_amt'] = train_df['card_proxy_id'].map(card_mean_amt_map)
val_df['card_mean_amt'] = val_df['card_proxy_id'].map(card_mean_amt_map)

# Fill unmapped new categories or missing data with training global baseline statistics
global_train_mean = train_df['TransactionAmt'].mean()
train_df['card_mean_amt'] = train_df['card_mean_amt'].fillna(global_train_mean)
val_df['card_mean_amt'] = val_df['card_mean_amt'].fillna(global_train_mean)

# Derive ratios
train_df['txn_amt_to_mean_card_ratio'] = train_df['TransactionAmt'] / train_df['card_mean_amt']
val_df['txn_amt_to_mean_card_ratio'] = val_df['TransactionAmt'] / val_df['card_mean_amt']

print("\n--- Engineering Rolling Velocity Window Sequences ---")
# Grouped frequencies derived cleanly over rolling context boundaries
# To compute cumulative transaction counts over runtime without leaks, rely on sequential cumcount blocks
train_df['txn_count_card_historical'] = train_df.groupby('card_proxy_id').cumcount()
# For the validation slice, we concatenate training history totals to align velocity profiles flawlessly
all_history_counts = train_df.groupby('card_proxy_id').size().to_dict()
val_df['txn_count_card_historical'] = val_df['card_proxy_id'].map(all_history_counts).fillna(0) + val_df.groupby('card_proxy_id').cumcount()

print("\n--- Executing Numeric Imputation Defense Layer ---")
# Isolate feature variables 
drop_cols = ['isFraud', 'TransactionID', 'TransactionDT', 'card_proxy_id']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

# Extract numerical column sets dynamically
num_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Generate median configurations locked strictly to training observations
train_medians = train_df[num_cols].median()

# Apply structural alignment updates
train_df[num_cols] = train_df[num_cols].fillna(train_medians)
val_df[num_cols] = val_df[num_cols].fillna(train_medians)

# Cache output features locally to accelerate Downstream Modeling
print("\nSerializing engineered artifacts cleanly to disk storage...")
DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data/processed')
train_df.to_parquet(os.path.join(DATA_PROCESSED, 'train_engineered.parquet'), index=False)
val_df.to_parquet(os.path.join(DATA_PROCESSED, 'val_engineered.parquet'), index=False)

print("Feature Engineering Phase Execution Finalized!")


--- Engineering Advanced Ratio Features ---

--- Engineering Rolling Velocity Window Sequences ---

--- Executing Numeric Imputation Defense Layer ---

Serializing engineered artifacts cleanly to disk storage...
Feature Engineering Phase Execution Finalized!


In [ ]:
# # 03_feature_engineering.py
# import pandas as pd
# import numpy as np
# import gc
# from src.data_utils import load_and_merge_data, engineer_time_features

# TRAIN_TXN = 'data/train_transaction.csv'
# TRAIN_ID = 'data/train_identity.csv'

# df = load_and_merge_data(TRAIN_TXN, TRAIN_ID)
# df = engineer_time_features(df)

# # Constructing card proxy identifier combinations to pinpoint distinct payment components securely
# print("Generating high-cardinality entity proxy tracking keys...")
# df['card_proxy_id'] = df['card1'].astype(str) + "_" + df['card2'].astype(str) + "_" + df['addr1'].astype(str)

# print("Enforcing strict chronological index split arrays...")
# df = df.sort_values('TransactionDT').reset_index(drop=True)

# # 70-30 Sequential Split Boundary Lookups
# split_idx = int(len(df) * 0.70)
# train_df = df.iloc[:split_idx].copy()
# val_df = df.iloc[split_idx:].copy()

# del df
# gc.collect()

# # =====================================================================
# # SYSTEM DEFENSE LOGIC: CALCULATION ENTIRELY ANCHORED TO TRAIN WINDOW
# # =====================================================================
# print("\n--- Engineering Advanced Ratio Features ---")
# # Group metrics computed STRICTLY on train records to fully block operational look-ahead leakages
# card_mean_amt_map = train_df.groupby('card_proxy_id')['TransactionAmt'].mean().to_dict()

# # Map the statistics safely to the cohorts
# train_df['card_mean_amt'] = train_df['card_proxy_id'].map(card_mean_amt_map)
# val_df['card_mean_amt'] = val_df['card_proxy_id'].map(card_mean_amt_map)

# # Fill unmapped new categories or missing data with training global baseline statistics
# global_train_mean = train_df['TransactionAmt'].mean()
# train_df['card_mean_amt'] = train_df['card_mean_amt'].fillna(global_train_mean)
# val_df['card_mean_amt'] = val_df['card_mean_amt'].fillna(global_train_mean)

# # Derive ratios
# train_df['txn_amt_to_mean_card_ratio'] = train_df['TransactionAmt'] / train_df['card_mean_amt']
# val_df['txn_amt_to_mean_card_ratio'] = val_df['TransactionAmt'] / val_df['card_mean_amt']

# print("\n--- Engineering Rolling Velocity Window Sequences ---")
# # Grouped frequencies derived cleanly over rolling context boundaries
# # To compute cumulative transaction counts over runtime without leaks, rely on sequential cumcount blocks
# train_df['txn_count_card_historical'] = train_df.groupby('card_proxy_id').cumcount()
# # For the validation slice, we concatenate training history totals to align velocity profiles flawlessly
# all_history_counts = train_df.groupby('card_proxy_id').size().to_dict()
# val_df['txn_count_card_historical'] = val_df['card_proxy_id'].map(all_history_counts).fillna(0) + val_df.groupby('card_proxy_id').cumcount()

# print("\n--- Executing Numeric Imputation Defense Layer ---")
# # Isolate feature variables 
# drop_cols = ['isFraud', 'TransactionID', 'TransactionDT', 'card_proxy_id']
# feature_cols = [c for c in train_df.columns if c not in drop_cols]

# # Extract numerical column sets dynamically
# num_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# # Generate median configurations locked strictly to training observations
# train_medians = train_df[num_cols].median()

# # Apply structural alignment updates
# train_df[num_cols] = train_df[num_cols].fillna(train_medians)
# val_df[num_cols] = val_df[num_cols].fillna(train_medians)

# # Cache output features locally to accelerate Downstream Modeling 
# print("\nSerializing engineered artifacts cleanly to disk storage...")
# train_df.to_parquet('data/train_engineered.parquet', index=False)
# val_df.to_parquet('data/val_engineered.parquet', index=False)

# print("Feature Engineering Phase Execution Finalized!")